# 🤖 Automated Audit Workpaper Summarizer

**AI Engineer Track**  |  Difficulty: **Hard**  |  Domain: **Audit / Assurance (Big-4 style)**

> 💯 Built with 100% free tools — no paid API keys required. Uses a free `call_llm()` helper that
> tries **Cerebras → Groq → local Ollama** in that order, so this notebook costs $0 to run.

---

## 🧩 Problem Statement

Auditors manually scan thousands of journal entries for anomalies. Build a tool that statistically flags suspicious entries (round numbers, weekend postings, Benford's Law deviations) and uses a free local LLM to write the audit workpaper narrative explaining each flagged anomaly in professional audit language.

## 📁 Dataset

**Any general ledger / journal entry export (date, account, debit, credit, description)**

Source: [https://www.kaggle.com/datasets/webdevbadger/journal-entry-testing-dataset](https://www.kaggle.com/datasets/webdevbadger/journal-entry-testing-dataset)

⚠️ **Note:** If the real dataset file isn't uploaded to this Colab session, the code below
automatically generates a small realistic sample dataset with the same structure — so every cell
still runs successfully end-to-end even before you upload the real data.


## 🛠️ Tools Used

`Python 3 | pandas | Cerebras/Groq (free tiers) + Ollama fallback via local HTTP (all free, no paid API) | numpy | reportlab`

## 🔑 Before You Run

This notebook will ask for a **free Cerebras API key** and a **free Groq API key** (both have
generous free tiers, no credit card needed). You can get keys at:
- Cerebras: https://cloud.cerebras.ai
- Groq: https://console.groq.com/keys

If you skip both (just press Enter), it'll try to use a local Ollama server instead — that only
works if you have Ollama running on your own machine, not inside Colab.

---

### ⚠️ Disclaimer
This notebook is for educational / portfolio purposes only. It does not constitute financial,
legal, or investment advice.

---


In [3]:
!pip install pandas numpy openai groq ollama reportlab --break-system-packages

In [4]:
import os
from getpass import getpass
import requests
from openai import OpenAI
import pandas as pd
import numpy as np
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet
from reportlab.lib.pagesizes import letter

# ---------------------------------------------------------
# STEP 0: set up our 3 free AI options - cerebras first, then groq, then local ollama
# we ask for the api keys once, then build one call_llm() function that tries all 3
# ---------------------------------------------------------
if not os.environ.get("CEREBRAS_API_KEY"):
    os.environ["CEREBRAS_API_KEY"] = getpass("Enter your Cerebras API key (press enter to skip): ")
if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass("Enter your Groq API key (press enter to skip): ")

cerebras_client = OpenAI(api_key=os.environ.get("CEREBRAS_API_KEY", ""), base_url="https://api.cerebras.ai/v1")
groq_client = OpenAI(api_key=os.environ.get("GROQ_API_KEY", ""), base_url="https://api.groq.com/openai/v1")
OLLAMA_URL = "http://localhost:11434/api/generate"

def call_llm(prompt, system=None):
    # if there's a system instruction, we just stick it on top of the prompt
    # since we're keeping this simple and not building a full messages list
    full_prompt = f"{system}\n\n{prompt}" if system else prompt
    messages = [{"role": "user", "content": full_prompt}]

    # try cerebras first, it's free and fast
    try:
        response = cerebras_client.chat.completions.create(model="gpt-oss-120b", messages=messages)
        return response.choices[0].message.content.strip()
    except Exception as e:
        print("cerebras failed:", e)

    # try groq next
    try:
        response = groq_client.chat.completions.create(model="openai/gpt-oss-120b", messages=messages)
        return response.choices[0].message.content.strip()
    except Exception as e:
        print("groq failed:", e)

    # last resort - ollama running locally, called with a plain http request
    # (no extra "ollama" package needed, just "requests" which colab already has)
    try:
        resp = requests.post(OLLAMA_URL, json={"model": "llama3", "prompt": full_prompt, "stream": False})
        resp.raise_for_status()
        return resp.json()["response"].strip()
    except Exception as e:
        print("ollama failed too:", e)
        return "AI call failed, all 3 options did not work"






In [5]:
# ---------------------------------------------------------
# STEP 1: load the journal entries, if the file's missing just build some sample rows
# so we can still run through the whole thing and see how it works
# ---------------------------------------------------------
def load_journal_entries(path="journal_entries.csv"):
    if os.path.exists(path):
        df = pd.read_csv(path)
        df["date"] = pd.to_datetime(df["date"])
        return df
    print(f"couldn't find {path}, making some sample journal entries instead so the code runs")
    np.random.seed(1)
    dates = pd.date_range("2025-01-01", periods=200, freq="D")
    return pd.DataFrame({
        "date": np.random.choice(dates, 200),
        "account": np.random.choice(["Cash", "AP", "AR", "Revenue", "Expenses"], 200),
        "debit": np.random.choice([0, 500, 1000, 2500, 5000], 200),
        "credit": np.random.choice([0, 500, 1000, 2500, 5000], 200),
        "description": ["sample entry"] * 200,
    })

df = load_journal_entries()
df["amount"] = df["debit"].fillna(0) - df["credit"].fillna(0)
df["abs_amount"] = df["amount"].abs()



couldn't find journal_entries.csv, making some sample journal entries instead so the code runs


In [6]:
# ---------------------------------------------------------
# STEP 2: some simple rules to catch obviously weird entries
# ---------------------------------------------------------
df["flag_round_number"] = (df["abs_amount"] % 1000 == 0) & (df["abs_amount"] > 0)
df["flag_weekend"] = df["date"].dt.dayofweek >= 5
df["flag_duplicate"] = df.duplicated(subset=["account", "abs_amount"], keep=False)



In [7]:
# ---------------------------------------------------------
# STEP 3: benford's law check - basically real financial numbers follow a
# predictable pattern of first digits, so we check how far off we are from that
# ---------------------------------------------------------
def first_digit(n):
    s = str(int(abs(n)))
    return int(s[0]) if s[0] != "0" else None

df["first_digit"] = df["abs_amount"].apply(first_digit)
observed = df["first_digit"].value_counts(normalize=True).sort_index() * 100
benford_expected = {d: np.log10(1 + 1 / d) * 100 for d in range(1, 10)}
benford_df = pd.DataFrame({
    "observed_pct": observed,
    "expected_pct": pd.Series(benford_expected)
}).fillna(0)
benford_df["deviation"] = (benford_df["observed_pct"] - benford_df["expected_pct"]).abs()
print(benford_df)

     observed_pct  expected_pct  deviation
1.0     21.604938     30.103000   8.498061
2.0     25.308642     17.609126   7.699516
3.0      0.000000     12.493874  12.493874
4.0     20.370370      9.691001  10.679369
5.0     32.716049      7.918125  24.797925
6.0      0.000000      6.694679   6.694679
7.0      0.000000      5.799195   5.799195
8.0      0.000000      5.115252   5.115252
9.0      0.000000      4.575749   4.575749


In [8]:

# ---------------------------------------------------------
# STEP 4: give every entry a risk score by adding up the flags
# ---------------------------------------------------------
df["risk_score"] = (
    df["flag_round_number"].astype(int) * 2 +
    df["flag_weekend"].astype(int) * 1 +
    df["flag_duplicate"].astype(int) * 2
)
top_risk = df.sort_values("risk_score", ascending=False).head(10)


In [9]:
# ---------------------------------------------------------
# STEP 5: for the riskiest entries, ask our free call_llm() to write the audit note
# ---------------------------------------------------------
def write_workpaper_note(row):
    reasons = []
    if row["flag_round_number"]: reasons.append("round-dollar amount")
    if row["flag_weekend"]: reasons.append("posted on a weekend")
    if row["flag_duplicate"]: reasons.append("duplicate amount/account combo")
    prompt = f"""You're an audit senior writing a workpaper note. This journal entry:
Date: {row['date'].date()}, Account: {row['account']}, Amount: {row['amount']:.2f}
Description: {row['description']}
was flagged because of: {', '.join(reasons)}
Write 2 simple sentences explaining why it was picked for testing and what to check next."""
    return call_llm(prompt)

top_risk = top_risk.copy()
top_risk["ai_note"] = top_risk.apply(write_workpaper_note, axis=1)



cerebras failed: Error code: 402 - {'message': 'Payment required to access this resource. Visit your billing tab.', 'type': 'payment_required_error', 'param': 'quota', 'code': 'payment_required'}
cerebras failed: Error code: 402 - {'message': 'Payment required to access this resource. Visit your billing tab.', 'type': 'payment_required_error', 'param': 'quota', 'code': 'payment_required'}
cerebras failed: Error code: 402 - {'message': 'Payment required to access this resource. Visit your billing tab.', 'type': 'payment_required_error', 'param': 'quota', 'code': 'payment_required'}
cerebras failed: Error code: 402 - {'message': 'Payment required to access this resource. Visit your billing tab.', 'type': 'payment_required_error', 'param': 'quota', 'code': 'payment_required'}
cerebras failed: Error code: 402 - {'message': 'Payment required to access this resource. Visit your billing tab.', 'type': 'payment_required_error', 'param': 'quota', 'code': 'payment_required'}
cerebras failed: Err

In [10]:
# ---------------------------------------------------------
# STEP 6: put the flagged entries and their notes into a pdf
# ---------------------------------------------------------
doc = SimpleDocTemplate("audit_workpaper_summary.pdf", pagesize=letter)
styles = getSampleStyleSheet()
story = [Paragraph("Audit Workpaper - Flagged Journal Entries", styles["Title"]), Spacer(1, 12)]
for _, row in top_risk.iterrows():
    story.append(Paragraph(f"<b>{row['date'].date()} | {row['account']} | {row['amount']:.2f}</b>", styles["Heading3"]))
    story.append(Paragraph(row["ai_note"], styles["Normal"]))
    story.append(Spacer(1, 10))
doc.build(story)
print("saved audit_workpaper_summary.pdf, all done!")


saved audit_workpaper_summary.pdf, all done!
